In [ ]:
#Codigo 1 - Verificar GPU
import subprocess

resultado = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if resultado.returncode == 0:
    print(resultado.stdout)
else:
    print('⚠️  No se detectó GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno > GPU')

Sun Aug  9 02:55:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#codigo 2 - Instalar YOLOv8Ultralytics
%pip install ultralytics --quiet

import ultralytics
ultralytics.checks()
print(f'\n✅ Ultralytics {ultralytics.__version__} instalado correctamente')

Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 46.7/112.6 GB disk)

✅ Ultralytics 8.4.116 instalado correctamente


In [ ]:
#Codigo 3 - Conectar a mi Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado en /content/drive')

Mounted at /content/drive
✅ Google Drive montado en /content/drive


In [ ]:
#Codigo 4 - Buscar las carpetas train, valid y test , y el data.yaml
import os

DATASET_PATH = '/content/drive/MyDrive/Dataset_Augmentation_Balanceado'
DATA_YAML    = os.path.join(DATASET_PATH, 'data.yaml')

carpetas_requeridas = ['train/images', 'train/labels',
                       'valid/images',  'valid/labels',
                       'test/images',   'test/labels']

print(f'📂 Ruta del dataset: {DATASET_PATH}\n')

# Verificar data.yaml
if os.path.exists(DATA_YAML):
    print(f'✅ data.yaml encontrado')
    with open(DATA_YAML, 'r') as f:
        print(f.read())
else:
    print(f'❌ data.yaml NO encontrado en {DATA_YAML}')

# Verificar carpetas y contar imágenes
print('\n📊 Conteo de archivos por carpeta:')
for carpeta in carpetas_requeridas:
    ruta = os.path.join(DATASET_PATH, carpeta)
    if os.path.exists(ruta):
        n = len([f for f in os.listdir(ruta)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.txt'))])
        print(f'  ✅ {carpeta}: {n} archivos')
    else:
        print(f'  ❌ {carpeta}: carpeta NO encontrada')

📂 Ruta del dataset: /content/drive/MyDrive/Dataset_Augmentation_Balanceado

✅ data.yaml encontrado
# Archivo de configuracion para YOLOv8 Segmentacion - Tesis (dataset balanceado)
path: /content/drive/MyDrive/Dataset_Augmentation_Balanceado
train: train/images
val: valid/images
test: test/images

nc: 7
names:
  0: Bourbon
  1: Castillo
  2: Geisha
  3: Mundonovo
  4: SanSalvador
  5: Sarchimor
  6: Typica


📊 Conteo de archivos por carpeta:
  ✅ train/images: 3819 archivos
  ✅ train/labels: 3819 archivos
  ✅ valid/images: 468 archivos
  ✅ valid/labels: 462 archivos
  ✅ test/images: 231 archivos
  ✅ test/labels: 231 archivos


In [ ]:
#Codigo 5 - Configurar Parametros de Entrenamiento
# ─── PARÁMETROS ──────────────────────────────────────────────────────
MODEL        = 'yolov8n-seg.pt'   # ← (antes era yolov8n.pt)
EPOCHS       = 50
BATCH_SIZE   = 16
IMG_SIZE     = 640
PROJECT_NAME = 'cafe_yolov8'
RUN_NAME     = 'entrenamiento_v1'
# ─────────────────────────────────────────────────────────────────────

print('⚙️  Configuración de entrenamiento:')
print(f'  • Modelo base  : {MODEL}')
print(f'  • Épocas       : {EPOCHS}')
print(f'  • Batch size   : {BATCH_SIZE}')
print(f'  • Tamaño imagen: {IMG_SIZE}px')
print(f'  • Proyecto     : {PROJECT_NAME}/{RUN_NAME}')

⚙️  Configuración de entrenamiento:
  • Modelo base  : yolov8n-seg.pt
  • Épocas       : 50
  • Batch size   : 16
  • Tamaño imagen: 640px
  • Proyecto     : cafe_yolov8/entrenamiento_v1


In [ ]:
# Código 6 - Entrenamiento + Evaluación + Gráficas + Predicciones + Guardado automático
import os
import random
import shutil
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO

# =============================================================
# 1) ENTRENAMIENTO
# =============================================================
model = YOLO(MODEL)

results = model.train(
    data    = DATA_YAML,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    project = PROJECT_NAME,
    name    = RUN_NAME,
    device  = 0,          # 0 = GPU, 'cpu' = CPU
    plots   = True,
    save    = True,
    verbose = True,
)

print('\n✅ Entrenamiento completado')
print(f'📁 Resultados guardados en: {PROJECT_NAME}/{RUN_NAME}')

BEST_MODEL = f'/content/runs/segment/{PROJECT_NAME}/{RUN_NAME}/weights/best.pt'



Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Dataset_Augmentation_Balanceado/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.

In [ ]:
# =============================================================
# 5) GUARDAR MODELO EN GOOGLE DRIVE (Ejecutar esta celda despues del entrenamiento para guardar el modelo)
# =============================================================

DEST_FOLDER = '/content/drive/MyDrive/Dataset_Augmentation_Balanceado/modelos_entrenados' #<- Podemos cambiar la ruta si es necesario
DEST_FILE   = os.path.join(DEST_FOLDER, 'best_cafe_yolov8_seg.pt')

os.makedirs(DEST_FOLDER, exist_ok=True)

if os.path.exists(BEST_MODEL):
    shutil.copy(BEST_MODEL, DEST_FILE)
    print(f'\n✅ Modelo guardado automáticamente en Google Drive:')
    print(f'   {DEST_FILE}')
else:
    print(f'\n❌ No se encontró el modelo en: {BEST_MODEL}')


✅ Modelo guardado automáticamente en Google Drive:
   /content/drive/MyDrive/Dataset_Augmentation_Balanceado/modelos_entrenados/best_cafe_yolov8_seg.pt
